In [29]:
import os
import torch
import librosa
import time
from scipy.io.wavfile import write
from tqdm import tqdm

import utils
from models import SynthesizerTrn
from mel_processing import mel_spectrogram_torch
from hyperpyyaml import load_hyperpyyaml
from ECAPA_TDNN.utils import encode_batch_vn
import logging

In [30]:
hpfile = "ECAPA_TDNN/config.json"
ptfile = "ECAPA_TDNN/G_80000.pth"
txtpath = "convert.txt"
outdir = "output/freevc"
use_timestamp = False

In [31]:
print("Loading speaker encoder...")
with open('ECAPA_TDNN/hyperparams.yaml', 'r', encoding='utf-8') as fin:
    params = load_hyperpyyaml(fin)
ecapa = params['embedding_model']
ecapa.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ecapa.to(device)
ckpt = torch.load('ECAPA_TDNN/embedding_model.ckpt', map_location=device)
ecapa.load_state_dict(ckpt, strict=False)

Loading speaker encoder...
INFO:speechbrain.utils.seed:Setting seed to 1986


<All keys matched successfully>

In [32]:
def get_emb(path):
    # 1) Load wav → numpy
    wav_np, sr = librosa.load(path, sr=16000)
    wav_np = wav_np.astype('float32')

    # 2) Chuyển sang tensor shape (1, T)
    wav = torch.from_numpy(wav_np).unsqueeze(0)

    # 3) Encode bằng đúng pipeline SpeechBrain
    g = encode_batch_vn(wav, ecapa, device, params)

    # 4) Trả về tensor 1-D
    return g.squeeze()


In [33]:
def map_to_gt_name(base_name):
    """
    Ví dụ:
    "F_B30_to_M_B30__S1-S2.wav" -> "M_S2_B30.wav"
    Hàm chấp nhận cả đường dẫn, sẽ lấy phần basename.
    Trả về None nếu không khớp.
    """
    import os, re
    bn = os.path.basename(base_name)

    # pattern đầy đủ, bắt các nhóm rõ ràng
    m = re.match(
        r'^(?P<src_g>[A-Z])_(?P<src_age>[A-Z]\d+)_to_(?P<tgt_g>[A-Z])_(?P<tgt_age>[A-Z]\d+)__(?P<src_spk>[A-Z]\d+)-(?P<tgt_spk>[A-Z]\d+)\.wav$',
        bn
    )
    if not m:
        return None

    tgt_g = m.group('tgt_g')      # M hoặc F
    tgt_age = m.group('tgt_age')  # B30, N60, ...
    tgt_spk = m.group('tgt_spk')  # S2, U1, ...

    return f"{tgt_g}_{tgt_spk}_{tgt_age}.wav"


In [35]:
base_txt = "output/freevc_ecapa_nonpre/_base.txt"
input_dir = "output/freevc_ecapa_nonpre"
gt_dir = "CER_WER"

base_files = []

with open(base_txt, "r", encoding="utf-8") as f:
    for line in f:
        if "|" not in line:
            continue
        fname, _ = line.strip().split("|", 1)
        base_files.append(fname.strip())

results = []
for fname in base_files:
    wav_path = os.path.join(input_dir, fname)
    gt_name = map_to_gt_name(fname)
    gt_path = os.path.join(gt_dir, gt_name)
    if not os.path.exists(gt_path):
        print("GT missing:", gt_path)
        continue
    emb_pred = get_emb(wav_path)
    emb_gt = get_emb(gt_path)

    score = F.cosine_similarity(emb_pred, emb_gt, dim=0).item()
    results.append((fname, gt_name, score))

scores = [x[2] for x in results]
avg = sum(scores) / len(scores)
print("---- Speaker Similarity Results ----")
for a, b, s in results:
    print(f"{a}  <->  {b}   | cos = {s:.4f}")
print("\nMean cosine similarity:", avg)

embeddings torch.Size([1, 1, 192])
embeddings torch.Size([1, 1, 192])
embeddings torch.Size([1, 1, 192])
embeddings torch.Size([1, 1, 192])
embeddings torch.Size([1, 1, 192])
embeddings torch.Size([1, 1, 192])
embeddings torch.Size([1, 1, 192])
embeddings torch.Size([1, 1, 192])
embeddings torch.Size([1, 1, 192])
embeddings torch.Size([1, 1, 192])
embeddings torch.Size([1, 1, 192])
embeddings torch.Size([1, 1, 192])
embeddings torch.Size([1, 1, 192])
embeddings torch.Size([1, 1, 192])
embeddings torch.Size([1, 1, 192])
embeddings torch.Size([1, 1, 192])
embeddings torch.Size([1, 1, 192])
embeddings torch.Size([1, 1, 192])
embeddings torch.Size([1, 1, 192])
embeddings torch.Size([1, 1, 192])
embeddings torch.Size([1, 1, 192])
embeddings torch.Size([1, 1, 192])
embeddings torch.Size([1, 1, 192])
embeddings torch.Size([1, 1, 192])
embeddings torch.Size([1, 1, 192])
embeddings torch.Size([1, 1, 192])
embeddings torch.Size([1, 1, 192])
embeddings torch.Size([1, 1, 192])
embeddings torch.Siz

In [ ]:
# ECAPA-TDNN_cyclic_loss: 0.497688855859451
# FreeVC_ECAPA: 0.46994419174734503
# FreeVC_VLSP: 0.277665947949572
# FreeVC: 0.24729042527178535